# 00: Environment Setup, Diagnostics & Canonical Data Acquisition
## ETHUSDT Quantitative Research & Signal Platform

This notebook serves as the primary initialization controller in Google Colab and local environments:
1. Clones/pulls latest code from GitHub and configures environment.
2. Mounts Google Drive persistent storage.
3. Detects hardware resources (CPU, RAM, GPU, CUDA).
4. Executes `quant doctor` diagnostic audit.
5. Downloads authoritative ETHUSDT 1m futures archives from Binance with SHA-256 validation.
6. Canonicalizes data into partitioned Parquet format.
7. Audits data integrity, continuity, and manifests.

In [ ]:
# =============================================================================
# 0. GOOGLE COLAB / LOCAL REPOSITORY SYNC & SETUP
# =============================================================================
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")
REPO_URL = "https://github.com/umutergul74/daytrader.git"
REPO_DIR = Path("/content/daytrader")

if IN_COLAB:
    print("🚀 [Google Colab Detected] Initializing Daytrader Platform...")
    if not REPO_DIR.exists():
        print(f"Cloning latest repository from {REPO_URL}...")
        !git clone {REPO_URL} /content/daytrader
    else:
        print("Pulling latest updates from GitHub...")
        !cd /content/daytrader && git pull

    os.chdir(str(REPO_DIR))
    print("Installing dependencies...")
    !pip install -q polars pandas numpy scipy scikit-learn lightgbm xgboost catboost pydantic pydantic-settings typer rich matplotlib pyarrow requests websockets pytest optuna

    if str(REPO_DIR / "src") not in sys.path:
        sys.path.insert(0, str(REPO_DIR / "src"))
    print(f"✓ Environment ready! Working directory: {Path.cwd()}")
else:
    project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    os.chdir(str(project_root))
    if str(project_root / "src") not in sys.path:
        sys.path.insert(0, str(project_root / "src"))
    print(f"✓ [Local Mode] Working directory: {Path.cwd()}")

from quant_platform.colab.hardware import ResourceDetector
from quant_platform.colab.drive_manager import DrivePersistenceManager
from quant_platform.cli.commands.doctor import run_doctor

print("✓ Quant Platform Imported Successfully!")

In [ ]:
# 1. Run System & Environment Diagnostics
run_doctor()

In [ ]:
# 2. Mount Google Drive and Initialize Layout
drive_mgr = DrivePersistenceManager()
if drive_mgr.mount_drive_in_colab():
    drive_mgr.init_drive_layout()
else:
    print("Running in Local / Non-Colab mode.")

In [ ]:
# 3. Fetch Canonical ETHUSDT 1-Minute Futures Data
from quant_platform.data.providers.binance_archive import BinancePublicArchiveProvider
from quant_platform.data.storage.canonical import CanonicalStorage

provider = BinancePublicArchiveProvider()
storage = CanonicalStorage()

# Fetch sample month (e.g. 2024-01)
SYMBOL = "ETHUSDT"
YEAR = 2024
MONTH = 1

print(f"Fetching {SYMBOL} 1m for {YEAR}-{MONTH:02d}...")
df_month = provider.fetch_month(SYMBOL, "1m", YEAR, MONTH)
if df_month is not None and not df_month.is_empty():
    part_path = storage.write_month_partition(df_month, year=YEAR, month=MONTH, symbol=SYMBOL)
    print(f"Saved {len(df_month)} bars to canonical partition: {part_path}")
else:
    print("Data fetch skipped or offline.")

In [ ]:
# 4. Data Integrity Audit & Continuity Check
from quant_platform.data.validation.integrity import DataIntegrityValidator

df_canonical = storage.read_range(symbol=SYMBOL, timeframe="1m")
if not df_canonical.is_empty():
    report = DataIntegrityValidator.validate_1m_series(df_canonical)
    print(f"Total 1m Bars: {report.total_rows}")
    print(f"Duplicates: {report.duplicate_count}")
    print(f"Invalid OHLC: {report.invalid_ohlc_count}")
    print(f"Gaps: {report.gap_count}")
    print(f"Valid: {report.is_valid}")
else:
    print("No canonical data present.")

In [ ]:
# 5. Generate Immutable Dataset Manifest & Fingerprint
from quant_platform.data.manifest.manifest_manager import DatasetManifestManager

if not df_canonical.is_empty():
    manifest_mgr = DatasetManifestManager()
    manifest = manifest_mgr.generate_manifest(symbol=SYMBOL, timeframe="1m")
    print(f"Dataset ID: {manifest.dataset_id}")
    print(f"SHA-256 Fingerprint: {manifest.canonical_fingerprint}")
    print("SYSTEM_READY: Environment and Canonical Data Verified!")